# VGG -- static pruning ladder (CIFAR-10)

| phase | optimizer | epochs | batch |
|---|---|---|---|
| dense pretrain | SGD 0.01, linear warmup | 100 (patience 20) | 512 |
| I.P. prune | SGD 0.01 | 5 + 10 recovery | 512 |
| BaCP | SGD 0.1, tau 0.15 | 5 + 10 recovery, then AdamW 1e-4 x 50 | 512 |

Protocol from the original paper (appendix Tables 1-2, Section 4.1). Static pruning at
0.95 / 0.97 / 0.99 from an **ImageNet-pretrained** model. Dynamic methods (RigL, EAST)
are **referenced from their papers**, not re-run here.

**Defect M2 is fixed in this commit**: previously the pruning scope excluded any parameter whose name contains `classifier`, which for VGG exempted the entire 119M-param MLP head -- a "99% sparse VGG-11" left 92.8% of the network dense. Any earlier VGG sparse number is void. VGG-19's BaCP phase uses LR **0.05** (gradient explosion at 0.1, appendix Table 1).


In [ ]:
import sys, pathlib

# Find nb_common.py whether the kernel started in this folder or at the repo root.
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')

import nb_common as nb
info = nb.setup()

## Configure

Everything below reads these knobs. `SMOKE=True` runs the whole pipeline on 2 batches to prove it end to end before any real GPU time.

In [ ]:
MODEL      = 'vgg19'      # vgg19 | vgg11
SEED       = 1
GPU        = 0
SPARSITIES = (0.95, 0.97, 0.99)
PRUNERS    = ('magnitude', 'snip', 'wanda')
SMOKE      = False    # True = 2-batch pipeline check; records get a .smoke key
OVERRIDES  = {}       # e.g. dict(epochs=3) to shorten every run below

for phase in ('dense', 'prune', 'bacp'):
    print(f'{phase:>6}: ', {**nb.FAMILIES[MODEL]['base'], **nb.FAMILIES[MODEL][phase]})

## Weights + preflight

The check whose absence cost a day of training: `load_weights` fails soft, so a run that asked for pretrained weights can silently train from random init. This cell makes that impossible.

In [ ]:
# Halts, loudly, if the model is not actually pretrained -- run this before
# spending any GPU time. fetch places real ImageNet weights at the exact path
# the registry expects; HF models (ViT/BERT) skip the fetch (hub weights).
nb.fetch_imagenet_weights(MODEL)
nb.preflight(MODEL, num_classes=nb.FAMILIES[MODEL]['base']['num_classes'])

## Dense baseline (required first)

The pretrained model finetuned on the task. Every sparse cell below starts from this checkpoint (same seed), so it must complete before anything else. Re-running skips it if a record exists.

In [ ]:
dense = nb.make_cell(MODEL, 'dense', seed=SEED, smoke=SMOKE, **OVERRIDES)
out = nb.run(dense, gpu=GPU)

## I.P. -- magnitude

Iterative pruning + recovery, the paper's sparse baseline. One run per sparsity level, streamed back to back.

In [ ]:
cells = [nb.make_cell(MODEL, 'prune', seed=SEED, pruner='magnitude', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## I.P. -- SNIP

In [ ]:
cells = [nb.make_cell(MODEL, 'prune', seed=SEED, pruner='snip', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## I.P. -- WANDA

In [ ]:
cells = [nb.make_cell(MODEL, 'prune', seed=SEED, pruner='wanda', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## BaCP -- magnitude

Contrastive pruning (PrC/SnC/FiC + CE, lambdas 0.25 each, tau 0.15), then AdamW 1e-4 finetune.

In [ ]:
cells = [nb.make_cell(MODEL, 'bacp', seed=SEED, pruner='magnitude', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## BaCP -- SNIP

In [ ]:
cells = [nb.make_cell(MODEL, 'bacp', seed=SEED, pruner='snip', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## BaCP -- WANDA

In [ ]:
cells = [nb.make_cell(MODEL, 'bacp', seed=SEED, pruner='wanda', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## Results vs the paper

`-` means not run yet (ours) or not published (paper).

In [ ]:
nb.results_table(MODEL)

## Health

Every static record on disk for this model. Delete a record under `results/runs/` to re-arm its cell.

In [ ]:
import runner as R
done = sorted(k for k in R.completed_keys() if k.startswith('static.') and MODEL in k)
print(f'{len(done)} static record(s) for {MODEL}:')
for k in done:
    print(' ', k)